In [0]:
df_silver = spark.read.table("hdfc_ergo.hdfc_silver.fact_labs")
df_dim_members = spark.read.table("hdfc_ergo.hdfc_gold.dim_members")
df_dim_providers = spark.read.table("hdfc_ergo.hdfc_gold.dim_providers")
df_dim_facilities = spark.read.table("hdfc_ergo.hdfc_gold.dim_facilities")
df_dim_date = spark.read.table("hdfc_ergo.hdfc_gold.dim_date")
df_dim_cpt = spark.read.table("hdfc_ergo.hdfc_gold.dim_cpt_codes")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, coalesce, lit, col

df_gold = df_silver.withColumn("lab_sk", monotonically_increasing_id() + 1)

df_gold = df_gold.join(df_dim_members.select(col("member_sk").alias("gold_member_sk"), "member_id"), on="member_id", how="left").withColumn("gold_member_sk", coalesce(col("gold_member_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_providers.select(col("provider_sk").alias("gold_provider_sk"), "provider_id"), on="provider_id", how="left").withColumn("gold_provider_sk", coalesce(col("gold_provider_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_facilities.select(col("facility_sk").alias("gold_facility_sk"), "facility_id"), on="facility_id", how="left").withColumn("gold_facility_sk", coalesce(col("gold_facility_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_date.select(col("date_sk").alias("gold_date_sk"), col("date").alias("order_date")), on="order_date", how="left").withColumn("gold_date_sk", coalesce(col("gold_date_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_cpt.select(col("cpt_sk").alias("gold_cpt_sk"), "cpt_code"), on="cpt_code", how="left").withColumn("gold_cpt_sk", coalesce(col("gold_cpt_sk"), lit(-1)))

df_gold = df_gold.select(
    "lab_sk", "lab_order_id",
    col("gold_member_sk").alias("member_sk"), col("gold_provider_sk").alias("provider_sk"), 
    col("gold_facility_sk").alias("facility_sk"), col("gold_date_sk").alias("order_date_sk"),
    col("gold_cpt_sk").alias("cpt_code_sk"),
    "test_name", "abnormal_flag", "critical_value", "lab_charge", "insurance_paid"
)

display(df_gold.limit(5))

In [0]:
df_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("hdfc_ergo.hdfc_gold.fact_labs")
print("✅ Successfully wrote Fact_Labs to Gold layer.")